# Análisis t-SNE — historial OpenFang (bonus Módulo 3)

Pipeline previo:

1. **TASK-128** — `extraer_jsonl.py` → `output/sesiones.parquet`
2. **TASK-129** — `vectorizar.py` → `output/vectores.npy` + `metadatos.parquet`
3. **Este notebook** — t-SNE 2D/3D, KMeans sobre embeddings, Plotly e interpretación de clusters.

Temas esperados en la interpretación: dudas de **medicación**, señales de **alarma**, conversaciones **cortas/agradecimiento** (textos de demo sin PHI).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PROYECTO_ROOT = NOTEBOOK_DIR.parents[1]
else:
    PROYECTO_ROOT = NOTEBOOK_DIR.parents[0]

if str(PROYECTO_ROOT) not in sys.path:
    sys.path.insert(0, str(PROYECTO_ROOT))

from src.openfang.reduccion_tsne import (
    MIN_SESIONES_TSNE,
    SesionesInsuficientesTsne,
    cargar_artefactos_vectorizacion,
    contar_sesiones,
    ejecutar_pipeline_visual,
    resumen_clusters_por_etiqueta,
    sesiones_insuficientes,
)

OUTPUT_DIR = PROYECTO_ROOT / "analisis_tsne" / "output"
FIXTURE_VECTORES = PROYECTO_ROOT / "tests" / "fixtures" / "tsne_vectores_demo.npy"
FIXTURE_METADATOS = PROYECTO_ROOT / "tests" / "fixtures" / "tsne_metadatos_demo.parquet"
RUTA_VECTORES = OUTPUT_DIR / "vectores.npy"
RUTA_METADATOS = OUTPUT_DIR / "metadatos.parquet"

print("Raíz proyecto:", PROYECTO_ROOT)
print("Salida gráficos:", OUTPUT_DIR)

In [ ]:
uso_demo = False
motivo_demo = ""

if RUTA_VECTORES.is_file() and RUTA_METADATOS.is_file():
    X, metadatos = cargar_artefactos_vectorizacion(RUTA_VECTORES, RUTA_METADATOS)
    n_sesiones = contar_sesiones(metadatos)
    print(f"Artefactos reales cargados: {X.shape[0]} vectores, {n_sesiones} sesiones únicas.")
    if sesiones_insuficientes(n_sesiones):
        print(
            f"ADVERTENCIA: menos de {MIN_SESIONES_TSNE} sesiones únicas; "
            "no se ejecutará t-SNE sobre datos reales."
        )
        uso_demo = True
        motivo_demo = "pocas sesiones en runtime"
else:
    print("No hay vectores/metadatos en output/; se usará fixture demo.")
    uso_demo = True
    motivo_demo = "artefactos de vectorización ausentes"

if uso_demo:
    if not FIXTURE_VECTORES.is_file() or not FIXTURE_METADATOS.is_file():
        raise FileNotFoundError(
            "Ejecute extraer_jsonl + vectorizar o genere tests/fixtures/tsne_*_demo"
        )
    X, metadatos = cargar_artefactos_vectorizacion(FIXTURE_VECTORES, FIXTURE_METADATOS)
    print(f"Fixture demo ({motivo_demo}): {X.shape[0]} vectores, "
          f"{contar_sesiones(metadatos)} sesiones (textos sintéticos, sin PHI).")

In [ ]:
try:
    resultado = ejecutar_pipeline_visual(X, metadatos, OUTPUT_DIR, random_state=42)
except SesionesInsuficientesTsne as exc:
    print("t-SNE omitido:", exc)
    resultado = None
else:
    print(
        f"t-SNE completado: k={resultado.n_clusters}, "
        f"perplexity={resultado.perplexity}, sesiones={resultado.n_sesiones}"
    )
    print("Exportados:", OUTPUT_DIR / "tsne_2d.png", OUTPUT_DIR / "tsne_3d.html")

In [ ]:
if resultado is not None:
    import plotly.express as px

    df_viz = pd.DataFrame(
        {
            "x": resultado.coords_2d[:, 0],
            "y": resultado.coords_2d[:, 1],
            "cluster": resultado.etiquetas.astype(str),
            "session_id": metadatos["session_id"].astype(str),
        }
    )
    fig_inline = px.scatter(
        df_viz,
        x="x",
        y="y",
        color="cluster",
        hover_name="session_id",
        title="Vista previa t-SNE 2D",
    )
    fig_inline.show()

    resumen = resumen_clusters_por_etiqueta(metadatos, resultado.etiquetas)
    for etiqueta, ejemplos in resumen.items():
        print(f"\nCluster {etiqueta}:")
        for ej in ejemplos:
            print(f"  - {ej}")

## Interpretación de clusters (informe / sustentación)

> Los ejemplos provienen de transcripciones **anonimizadas o sintéticas** para la demo académica. No incluir PHI en el informe final.

### Cluster A — Dudas sobre medicación y horarios

Sesiones donde el paciente pregunta por **dosis**, **horarios** o **interacciones** con medicamentos del protocolo postoperatorio. En el espacio de embeddings quedan cercanas entre sí; el agente debe priorizar respuestas alineadas al corpus ingerido (RAG) y recordar límites del disclaimer.

Ejemplos típicos (demo): «¿A qué hora debo tomar el analgésico recetado?», «¿Puedo tomar ibuprofeno con el omeprazol del protocolo?»

### Cluster B — Señales de alarma o malestar relevante

Turnos con vocabulario de **urgencia**: fiebre alta, sangrado, dolor torácico, empeoramiento de la herida. Visualmente separados del resto; en producción conviene cruzar con guardrails/KV del Hand y escalamiento humano (alcance TAAM en `proyecto-2`).

Ejemplos típicos (demo): «Tengo fiebre de 38.5 y el sitio de la herida está muy rojo», «Siento opresión en el pecho y mareo fuerte».

### Cluster C — Conversaciones cortas y agradecimiento

Interacciones **breves** de cierre o agradecimiento, con poca carga clínica. Útiles para medir ruido en el historial y sesiones «resueltas» sin nueva intención.

Ejemplos típicos (demo): «Muchas gracias Bot Lili», «Ok entendido, gracias».

### Limitaciones

- t-SNE preserva **vecindades locales**, no distancias globales entre clusters.
- Con pocas sesiones reales (`< 20–30`) la forma del gráfico puede cambiar entre ejecuciones; usar más tráfico Telegram antes de la sustentación.
- KMeans opera en el espacio de **embeddings**; el color en el gráfico es coherente con intención semántica, no con coordenadas t-SNE.

## Alternativa opcional: UMAP

La rúbrica permite **UMAP** en lugar de (o además de) t-SNE. El paquete `umap-learn` ya está en `pyproject.toml`. Ejemplo (no ejecutado por defecto):

```python
# import umap
# reducer = umap.UMAP(n_components=2, random_state=42)
# coords_umap = reducer.fit_transform(X)
```